# Verification

**Prerequisites:** `01_what_is_a_tool.ipynb`, `03_invocation.ipynb`

Everything checked so far happens on our side of the call.

Notebook 1 checks the arguments before the tool runs. Notebook 3 sorts out how the call ended — it returned, it raised, it took too long. Both are about the call. Neither one ever looks at the world the tool was supposed to change.

So a tool that runs cleanly, returns a cheerful string, and did not do the job counts as a success. The agent then builds its next three steps on something that never happened.

**By the end you'll have:**
- Watched a tool report success while quietly saving less than half of what it was given
- Written a check that reads the tool's own answer, and seen it catch nothing at all
- Built `verified()` in `verify.py`, which wraps a tool so the world gets checked after it runs
- Added a fifth ending to `invoke`: `unverified`, which is not the same as a tool that says no
- Written the cheaper kind of check, for when reading everything back is not an option
- Been honest about the calls that cannot be verified, and what to claim instead

## Setup

In [1]:
import sys
from pathlib import Path
from typing import Annotated

sys.path.append(str(Path.cwd().parent))
sys.path.append(str(Path.cwd().parent.parent / "01_foundations"))
sys.path.append(str(Path.cwd().parent.parent / "02_agent_runtime"))

from invoke import invoke
from registry import Registry
from tools import tool
from verify import VerificationError, verified

## A tool that says it worked

Here is a note store with a size limit, written the way plenty of real systems are written: the limit is enforced, and nobody is told about it.

In [2]:
NOTES = {}
LIMIT = 60

@tool(repeatable=False)
def save_note(name: Annotated[str, "What to file the note under"],
              text: Annotated[str, "The note itself"]) -> str:
    """Save a note so it can be read back later."""
    NOTES[name] = text[:LIMIT]
    return f"Saved '{name}'."

long_note = ("Deploy is Friday 14:00 UTC. Roll back with make rollback. "
             "Ping Priya if the error rate goes above two percent.")

result = invoke(save_note, {"name": "deploy", "text": long_note})
print(result.kind, "|", result)
print()
print("asked to save: ", len(long_note), "characters")
print("actually there:", len(NOTES["deploy"]), "characters")
print(repr(NOTES["deploy"]))

ok | Saved 'deploy'.

asked to save:  110 characters
actually there: 60 characters
'Deploy is Friday 14:00 UTC. Roll back with make rollback. Pi'


`ok`. And there is nothing wrong with that verdict, given what `invoke` is allowed to look at.

The arguments were valid, so notebook 1's check passed. The function returned instead of raising, so notebook 3 had nothing to sort out. `save_note` even told the truth as it understood it — a note *was* saved under that name.

The rollback instructions are gone, and so is Priya. The agent will carry on as though the whole thing were filed, and every later step that depends on it is now wrong.

Validation asks *can this call be made*. Nothing so far asks *did it happen*.

## Checking the return value is not verification

The obvious first attempt is to look harder at what the tool gave back:

In [3]:
NOTES.clear()

looks_fine = verified(save_note,
                      lambda args, value: None if "Saved" in value else "the note was not saved")

result = invoke(looks_fine, {"name": "deploy", "text": long_note})
print(result.kind, "|", result)
print("still only:", len(NOTES["deploy"]), "characters")

ok | Saved 'deploy'.
still only: 60 characters


It passes, and it was never going to do anything else.

The report and the thing being reported come from the same place. `save_note` believes it saved the note — it has no idea the store trimmed it — so its answer is confident and wrong. A check reading that answer inherits both. A tool that is wrong about what it did will be wrong about it consistently.

**A check is worth something only if it looks somewhere the tool is not reporting from.** For a note store, that means reading the note back.

## Reading it back

In [4]:
NOTES.clear()

def note_was_stored_whole(args, value):
    stored = NOTES.get(args["name"])
    if stored is None:
        return f"Nothing is filed under '{args['name']}'. The note was not saved."
    if stored != args["text"]:
        return (f"Only the first {len(stored)} characters of {len(args['text'])} were saved "
                f"under '{args['name']}'. It now reads: {stored!r}")
    return None

checked_save = verified(save_note, note_was_stored_whole)

result = invoke(checked_save, {"name": "deploy", "text": long_note})
print(result.kind)
print(result)
print()
print("goes to the model:", result.for_the_model, "| safe to retry:", result.safe_to_retry)

unverified
Only the first 60 characters of 110 were saved under 'deploy'. It now reads: 'Deploy is Friday 14:00 UTC. Roll back with make rollback. Pi'

goes to the model: True | safe to retry: False


`verified()` is a wrapper, and a small one:

```python
def verified(tool, check):
    def call(**kwargs):
        value = tool.function(**kwargs)
        problem = check(kwargs, value)
        if problem:
            raise VerificationError(problem)
        return value

    checked = copy.copy(tool)
    checked.function = call
    return checked
```

A copy, not a change to the original. The same tool may be registered somewhere else, or handed to another agent that checks it differently, or not at all.

The schema is untouched, because `.schema` is built from `name` and `args`, which the copy keeps. The model is not told that its work is being checked, and there is nothing useful it could do with that information anyway. **Verification is something the caller does about the answer, not part of the deal the tool offers.**

In [5]:
print("same schema as before:", checked_save.schema == save_note.schema)

NOTES.clear()
short = invoke(checked_save, {"name": "deploy", "text": "Deploy is Friday."})
print("a note that fits:", short.kind, "->", repr(NOTES["deploy"]))

same schema as before: True
a note that fits: ok -> 'Deploy is Friday.'


### Why `unverified` is its own ending

`invoke` gained a fifth kind, next to the four from notebook 3:

```python
except ToolError as e:
    return Result(tool.name, "refused", str(e), ...)
except VerificationError as e:
    return Result(tool.name, "unverified", str(e), ...)
```

Those two look similar and they mean opposite things.

**`refused`** is a tool that ran, decided it could not do the job, and said so. Nothing happened. The world is exactly as it was.

**`unverified`** is a tool that said it *had* done the job. So something probably did happen — just not the thing that was reported. Half a note is filed. The world moved, and nobody can say how far.

That is why `safe_to_retry` treats it like a timeout rather than like a refusal:

```python
if self.kind in ("timeout", "unverified"):
    return self.repeatable
```

`save_note` is `repeatable=False`, so trying again is not automatically safe. The state is already unknown, and another call may make it stranger rather than better.

The message itself goes to the model, because the model is about to build on something that did not happen. It is written as a fact — *only the first 60 characters were saved, and here is what it now reads* — rather than as a complaint. The model can act on the first and not on the second.

## When reading it back is too expensive

Comparing the whole result works for a note. It does not work when the thing you would have to read back is a million rows, or a bank statement, or anything that costs money to fetch.

The way out is that you rarely need the whole world. You need one thing that has to be true afterwards:

In [6]:
BALANCE = {"acme": 500.0}

@tool(repeatable=False)
def issue_refund(account: Annotated[str, "Account name"],
                 amount: Annotated[float, "How much to refund"]) -> str:
    """Refund money to a customer account."""
    BALANCE[account] -= amount / 10        # the bug: refunds a tenth of what it reports
    return f"Refunded {amount} to {account}."

def balance_moved_by_the_amount(args, value):
    moved = before[args["account"]] - BALANCE[args["account"]]
    if abs(moved - args["amount"]) > 0.001:
        return (f"The balance for {args['account']} moved by {moved}, "
                f"not the {args['amount']} that was reported as refunded.")
    return None

before = dict(BALANCE)
result = invoke(verified(issue_refund, balance_moved_by_the_amount),
                {"account": "acme", "amount": 50.0})
print(result.kind, "|", result)

unverified | The balance for acme moved by 5.0, not the 50.0 that was reported as refunded.


One number, read once, catching a bug that no amount of careful reading of the tool's own output would find. `"Refunded 50.0 to acme."` is exactly what that tool was supposed to say.

Checks like this are usually cheaper than reading everything back, and they often catch more, because they describe what the call was *for* rather than what it returned.

## What can't be verified

`send_email` cannot be verified. You cannot read the recipient's inbox, and nothing you have access to will tell you the message arrived.

What you can do is check one step short of the claim: the mail server accepted it, an id came back, the outbox is empty. That is worth doing. It is also worth being exact about what it proves. The gap between *the queue took it* and *she read it* is where a confident agent tells someone their email was sent, an hour before it bounces.

Two rules follow, and both are mostly about wording:

- A check should say what it actually established. "The mail server accepted the message and returned id abc123" is a fact. "Email sent to Priya" is a guess dressed as one.
- A tool that cannot be verified should not be given a check that pretends otherwise. Leaving it unverified is honest. A `verified()` wrapper reading the return value is worse than nothing, because it looks like the job was done.

Checking an agent's claims against reality across a whole run, rather than one call at a time, is `11_evaluation`. Stopping a call that should never happen is `10_guardrails`.

## Into the loop

The same verified tool, in chapter 2's ReAct loop. The note is longer than the store will keep, so the model is told in the middle of its own run that what it just did did not work:

In [7]:
from chat import Conversation
from runtime import Runtime
from observe import Observe
from think import Think
from act import Act

NOTES.clear()
notes = Registry([checked_save])

runtime = Runtime(
    loop=[
        Observe(event="Save a note called 'deploy' with this text: Deploy is Friday 14:00 UTC. "
                      "Roll back with make rollback. Ping Priya if the error rate goes above "
                      "two percent."),
        Think(tools=notes.tools, allow_fork=False),
        Act(functions=notes.invokers(timeout=10), schemas=None),
        Observe(),
    ],
    repeat_from=1,
    max_iterations=8,
)

conversation = runtime.run(Conversation())
for m in conversation.messages:
    print(f"- {m['role']:9} {m['content']!r}")

print()
print("what ended up stored:", repr(NOTES.get("deploy")))

- user      "Save a note called 'deploy' with this text: Deploy is Friday 14:00 UTC. Roll back with make rollback. Ping Priya if the error rate goes above two percent."
- decision  '{"name":"deploy","text":"Deploy is Friday 14:00 UTC. Roll back with make rollback. Ping Priya if the error rate goes above two percent."}'
- tool      'Only the first 60 characters of the note were saved under “deploy”, and it now reads: “Deploy is Friday 14:00 UTC. Roll back with make rollback. Pi”.'
- decision  'Only the first 60 characters of the note were saved under “deploy”, and it now reads: “Deploy is Friday 14:00 UTC. Roll back with make rollback. Pi”.'
- assistant 'Only the first 60 characters of the note were saved under “deploy”, and it now reads: “Deploy is Friday 14:00 UTC. Roll back with make rollback. Pi”.'

what ended up stored: 'Deploy is Friday 14:00 UTC. Roll back with make rollback. Pi'


The note is still truncated. Verification does not fix anything — it notices. What changed is what the agent believes, and what it tells you.

Without the check, this run ends with the model reporting that the note was saved, because that is what the tool said. With it, the run ends with the model saying which sixty characters actually made it, and the rollback instructions are visibly missing rather than invisibly missing.

Two things in that transcript are worth reading closely.

**The message arrived intact.** `Observe` rephrases a raw tool result into an observation, and it kept both facts: how much was saved, and what the note now reads. That is the prompt change notebook 3 forced on chapter 2, doing its job. A verification message is exactly the kind of result whose useful half is easy to summarize away.

**The model reported rather than retried.** It did not try again with a shorter note, and that is defensible. Calling the same tool again truncates again, there is no tool for appending, and splitting the note across two names would mean inventing a name nobody asked for. `safe_to_retry` was `False` for the same reason.

So the check did not buy a fix. It bought the difference between an agent that is wrong and an agent that knows it.

## What you built

✓ Watched a tool save 60 characters of a 110-character note, return `"Saved 'deploy'."`, and come back from `invoke` as `ok` — because every check up to now looks at the call, not at the world

✓ Wrote a check that read the tool's own answer and caught nothing, for a reason worth keeping: the report and the thing reported come from the same place, so a tool that is wrong about what it did is wrong about it consistently

✓ Built `verified()` in `verify.py` — a copy of the tool with a check wrapped around its function, leaving the schema untouched, because whether an answer gets checked is the caller's business and not something the model can act on

✓ Added `unverified` to `invoke` as its own ending, and kept it separate from `refused`: a refusal means nothing happened, an unverified success means something did and nobody knows how much

✓ Made `safe_to_retry` treat `unverified` the way it treats a timeout, following the tool's own `repeatable` flag, because the state is already unknown

✓ Wrote the cheaper kind of check for when reading everything back is not an option — one thing that has to be true afterwards, which caught a refund paying out a tenth of what it reported

✓ Said plainly which calls cannot be verified, and what to claim instead: that the mail server accepted the message, not that it arrived

**Next:** `05_built_in_tools.ipynb` — the toolbox almost every agent ends up with: reading and writing files, fetching a page, running a search, doing arithmetic. Each one carrying a hazard worth meeting once, deliberately, rather than in production.